# Practice — KNN on `mpg.csv`

Same six stages as the Titanic notebook, on your own.

**Question:** given a car's specifications, was it built in the USA, Europe or Japan?

Target: `origin` &nbsp;·&nbsp; File: `mpg.csv`

In [49]:
import pandas as pd

df = pd.read_csv('/Users/tishikaagarwal/Desktop/Machine Learning/Assignments/mpg.csv')
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino


---
## 1. Look at the data

> **Flow:** Shape, columns, what is missing.

Run `.info()` and `.shape`. Which column has missing values, and how many?

In [50]:
print(df.info())
print(df.shape)


<class 'pandas.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        398 non-null    int64  
 5   acceleration  398 non-null    float64
 6   model_year    398 non-null    int64  
 7   origin        398 non-null    str    
 8   name          398 non-null    str    
dtypes: float64(4), int64(3), str(2)
memory usage: 28.1 KB
None
(398, 9)


How many cars from each `origin`? Use `value_counts()`.

In [51]:
df['origin'].value_counts()


origin
usa       249
japan      79
europe     70
Name: count, dtype: int64

---
## 2. Stage 1 — Data Cleaning

> **Flow:** Fix missing values, drop unusable columns.

Two jobs:

1. `horsepower` has blanks. Fill them with the median.
2. Drop `name`. In one line below, say why it cannot help the model.

In [52]:
df['horsepower'] = df['horsepower'].fillna(df['horsepower'].median())
df = df.drop(columns='name')

print(df.shape, "|" , df.isna().sum().sum() , "missing")
df.head()


(398, 8) | 0 missing


,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
0,18.0,8,307.0,130.0,3504,12.0,70,usa
1,15.0,8,350.0,165.0,3693,11.5,70,usa
2,18.0,8,318.0,150.0,3436,11.0,70,usa
3,16.0,8,304.0,150.0,3433,12.0,70,usa
4,17.0,8,302.0,140.0,3449,10.5,70,usa


*Why `name` cannot help:*

---
## 3. Features and Target

> **Flow:** `X` is everything the model looks at. `y` is the answer.

In [53]:
X = df.drop(columns="origin")
y = df['origin']

print(X.shape, y.shape)

(398, 7) (398,)


---
## 4. Stage 2 — Train/Test Split

> **Flow:** Hide some rows before preparing anything.

Use `test_size=0.2`, `random_state=0`, `stratify=y`.

In [54]:
from sklearn.model_selection import train_test_split

X_train , X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state=0, stratify=Y
)

print(X_train.shape, X_test.shape)

(318, 7) (80, 7)


---
## 5. Stage 3 — Feature Engineering

> **Flow:** Put every column on the same scale.

**No encoding needed here.** After dropping `name`, every feature is already a number,
so there is no text column left for `OneHotEncoder`. That happens in real projects too.

Print the min and max of each feature. Which column has the largest range?

In [55]:

X_train.describe().loc[['min', 'max']]

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year
min,9.0,3.0,68.0,46.0,1613.0,8.0,70.0
max,46.6,8.0,455.0,225.0,5140.0,24.8,82.0


Now scale. `StandardScaler` — `fit_transform` on train, `transform` on test.

In [56]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape, X_test_scaled.shape)


(318, 7) (80, 7)


---
## 6. Stages 4 and 5 — Train and Predict

> **Flow:** `.fit()` learns, `.predict()` answers.

**First without scaling.** Train `KNeighborsClassifier()` on the unscaled data and print the accuracy.

In [57]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

knn = KNeighborsClassifier()
knn.fit(X_train, y_train)

acc_raw = accuracy_score(y_test, knn.predict(X_test))
print(acc_raw)


0.6875


**Now with scaling.** Same model, same `k`, scaled data.

In [58]:
knn_scaled = KNeighborsClassifier()
knn_scaled.fit(X_train_scaled, y_train)

acc_scaled = accuracy_score(y_test, knn_scaled.predict(X_test_scaled))
print(acc_scaled)

0.75


---
## 7. Stage 6 — Compare

> **Flow:** Two numbers, one difference.

Print both accuracies together. Which is higher, and by how much?

In [59]:
print("without scaling:", round(acc_raw, 4))
print("with scaling:   ", round(acc_scaled, 4))
print("difference:     ", round(acc_scaled - acc_raw, 4))


without scaling: 0.6875
with scaling:    0.75
difference:      0.0625


*What changed between the two runs:*

---
## 8. Choosing k

> **Flow:** `k` is a dial. Try a few settings and look.

Run a loop over `k = 1, 3, 5, 7, 9, 11, 15, 21` on the **scaled** data.
Print `k` and its accuracy on each line.

In [60]:
for k in [1, 3, 5, 7, 9, 11, 15, 21]:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train_scaled, y_train)
    print(k, round(accuracy_score(y_test, m.predict(X_test_scaled)), 4))


1 0.7375
3 0.75
5 0.75
7 0.75
9 0.75
11 0.725
15 0.7375
21 0.725


*Best k:* &nbsp;&nbsp; *Its accuracy:*

Does the accuracy change a lot across k, or stay roughly flat?

*Your answers:*